Uganda Tax Engine: An Object-Oriented Tax Calculation System

### 1. Introduction

The Uganda Tax Engine is a Python-based mini-project that demonstrates how object-oriented programming can be applied to the calculation of selected taxes and payroll deductions.

The system implements reusable components for Pay As You Earn (PAYE), payroll deductions, Value Added Tax (VAT), and corporate income tax. It uses Python's `Decimal` arithmetic to improve monetary precision and configurable, effective-dated schedules to organize tax rules.

The project also demonstrates software engineering practices through custom exceptions, unit testing, property-based testing, and an executable Jupyter Notebook.

### 1.2. Project Objectives

The objectives of the project are to:

1. Develop an object-oriented tax calculation system using classes, inheritance, abstraction, and encapsulation.
2. Implement PAYE calculations for resident and nonresident taxpayers using configurable tax schedules.
3. Calculate employee and employer National Social Security Fund (NSSF) contributions, Local Service Tax, PAYE, and employee net pay.
4. Implement VAT calculations for standard-rated, zero-rated, and exempt supplies.
5. Calculate corporate income tax based on revenue and allowable deductions.
6. Use decimal-based monetary calculations, explicit rounding, and custom exceptions to improve reliability.
7. Validate the system through unit tests, boundary tests, and property-based testing.
8. Demonstrate the system's functionality and critically discuss its design decisions and limitations.

### 1.3. Scope and Assumptions

This project focuses on selected tax calculations for educational purposes. The implemented rates, effective dates, deductions, and other assumptions must be checked against applicable Uganda Revenue Authority guidance before the system is used for actual tax compliance.

The system is not intended to replace official tax computation or filing systems.

2: Import the tax engine modules

connect the notebook to the Python classes WE have already developed and tested.

In [24]:
from pathlib import Path
import sys

# Locate the project root by finding the src folder
current_path = Path.cwd()
project_root = None

for path in [current_path, *current_path.parents]:
    if (path / "src" / "uganda_tax_engine").is_dir():
        project_root = path
        break

if project_root is None:
    raise FileNotFoundError(
        "Could not locate the project root containing "
        "'src/uganda_tax_engine'."
    )

# Add the project root to Python's import path
sys.path.insert(0, str(project_root))

print("Project root:", project_root)
print("Import path configured successfully.")

Project root: c:\Users\GodfreyAkampa\OneDrive - raisingthevillage.org\12 OCTOBER 2026\Mini-Project 2 Uganda Tax Engine
Import path configured successfully.


2B: Import the tax engine

In [25]:
# Standard library imports
from datetime import date
from decimal import Decimal

# Tax engine imports
from src.uganda_tax_engine.money import Money
from src.uganda_tax_engine.core import TaxContext, TaxResult, Tax
from src.uganda_tax_engine.paye import PAYE
from src.uganda_tax_engine.payroll import Payroll
from src.uganda_tax_engine.vat import VAT
from src.uganda_tax_engine.corporate_tax import CorporateTax
from src.uganda_tax_engine.exceptions import (
    TaxError,
    InvalidIncomeError,
    NoScheduleForDateError,
)


3: OOP architecture

### 4. Object-Oriented Programming Architecture

The Uganda Tax Engine applies key object-oriented programming principles:

### 4.1 Abstraction
The abstract `Tax` class defines a common interface through the `calculate()` method. Each specific tax class implements its own calculation logic.

### 4.2 Inheritance
The `PAYE`, `VAT`, and `CorporateTax` classes inherit from the common `Tax` base class, promoting code reuse and a consistent interface.

### 4.3 Encapsulation
The `Money` class encapsulates monetary amounts and controls how they are created, rounded, compared, and manipulated. It uses `Decimal` rather than binary floating-point arithmetic.

### 4.4 Polymorphism
Different tax classes implement the same `calculate()` method but produce different results depending on the tax being calculated.

### 4.5 Supporting data structures
`TaxContext` carries the information required for a calculation, while `TaxResult` standardizes the result returned by tax classes. Custom exceptions provide explicit error handling.

In [26]:
# Verify inheritance and abstraction

tax_calculators = [
    PAYE(),
    VAT(),
    CorporateTax(),
]

for calculator in tax_calculators:
    print(
        f"{calculator.__class__.__name__} "
        f"is a Tax subclass: {isinstance(calculator, Tax)}"
    )

print("\nAll tax calculators expose the calculate method:")
for calculator in tax_calculators:
    print(
        f"{calculator.__class__.__name__}: "
        f"{callable(calculator.calculate)}"
    )

PAYE is a Tax subclass: True
VAT is a Tax subclass: True
CorporateTax is a Tax subclass: True

All tax calculators expose the calculate method:
PAYE: True
VAT: True
CorporateTax: True


4: the Money class and decimal precision

Here we’ll demonstrate how our Money value object handles UGX amounts safely and supports the operator overloading 

### 5. Monetary Representation and Precision

Financial calculations require careful handling of decimal values. Python's built-in `float` type uses binary floating-point representation, which can introduce small precision errors in some decimal calculations.

The tax engine uses Python's `Decimal` type and a custom `Money` class to represent Ugandan shilling amounts. The `Money` class rounds monetary values to two decimal places using `ROUND_HALF_UP` and rejects floating-point inputs.

It also overloads operators such as `__add__`, `__mul__`, `__lt__`, and `__repr__`, allowing monetary values to be added, multiplied, compared, and represented consistently.

In [27]:
# Demonstrate Money operations

salary = Money.from_value("1500000.50")
allowance = Money.from_value("250000.25")

total_income = salary + allowance
employee_contribution = total_income * Decimal("0.05")

print("Salary:", salary)
print("Allowance:", allowance)
print("Total income:", total_income)
print("5% contribution:", employee_contribution)
print("Is salary less than total income?", salary < total_income)

Salary: Money(UGX 1500000.50)
Allowance: Money(UGX 250000.25)
Total income: Money(UGX 1750000.75)
5% contribution: Money(UGX 87500.04)
Is salary less than total income? True


the danger of float

In [28]:
# Compare binary floating-point with decimal arithmetic

float_result = 0.1 + 0.2
decimal_result = Decimal("0.1") + Decimal("0.2")

print("Float result:", float_result)
print("Decimal result:", decimal_result)
print("Exact decimal result:", decimal_result == Decimal("0.3"))

Float result: 0.30000000000000004
Decimal result: 0.3
Exact decimal result: True


This illustrates why Decimal is useful for financial computations. It does not automatically guarantee that every tax calculation is correct; the formulas, rounding rules, and applicable tax schedules must also be correct.

5: PAYE calculations

demonstrates how  PAYE class calculates monthly income tax for resident employees at the required tax-band boundaries.

#### 6. Pay As You Earn (PAYE)

Pay As You Earn (PAYE) is a tax deducted from employment income. The tax engine implements separate calculation schedules for resident and nonresident taxpayers.

The calculation uses the taxpayer's income, applicable allowances, residency status, and effective date to determine taxable income and select the relevant schedule.

Testing values at tax-band boundaries is important because errors can occur where one marginal rate ends and another begins.

**Note:** The rates below reflect the project's supplied specification and can change or verified against current [Uganda Revenue Authority](https://ura.go.ug/en/changes-to-paye-return-form-following-the-income-tax-amendment-act-2026/) guidance.

In [29]:
from decimal import Decimal

# Required PAYE boundary cases from the project specification
paye_test_cases = [
    (335000, "0.00"),
    (335001, "0.20"),
    (410000, "15000.00"),
    (410001, "15000.25"),
    (485000, "33750.00"),
    (485001, "33750.30"),
    (10000000, "2888250.00"),
    (10000001, "2888250.40"),
]

paye_calculator = PAYE()

for income, expected_tax in paye_test_cases:
    context = TaxContext(
        income=Money.from_value(income),
        effective_date=date(2026, 10, 18),
        residency_status="resident",
    )

    result = paye_calculator.calculate(context)

    print(
        f"Income: UGX {income:>10,} | "
        f"PAYE: UGX {result.amount.amount:>12,.2f} | "
        f"Expected: UGX {Decimal(expected_tax):>12,.2f}"
    )

    assert result.amount.amount == Decimal(expected_tax)

print("\nAll resident PAYE boundary checks passed.")

Income: UGX    335,000 | PAYE: UGX         0.00 | Expected: UGX         0.00
Income: UGX    335,001 | PAYE: UGX         0.20 | Expected: UGX         0.20
Income: UGX    410,000 | PAYE: UGX    15,000.00 | Expected: UGX    15,000.00
Income: UGX    410,001 | PAYE: UGX    15,000.25 | Expected: UGX    15,000.25
Income: UGX    485,000 | PAYE: UGX    33,750.00 | Expected: UGX    33,750.00
Income: UGX    485,001 | PAYE: UGX    33,750.30 | Expected: UGX    33,750.30
Income: UGX 10,000,000 | PAYE: UGX 2,888,250.00 | Expected: UGX 2,888,250.00
Income: UGX 10,000,001 | PAYE: UGX 2,888,250.40 | Expected: UGX 2,888,250.40

All resident PAYE boundary checks passed.


These assertions validate the implemented formulas against the project's expected values. They do not establish that the schedule is legally current.

6: nonresident PAYE and effective-dated schedules

shows how the tax engine handles different residency statuses and checks whether a tax schedule applies on a particular date.

#### 7. Residency Status and Effective-Dated Tax Schedules

The tax engine maintains separate PAYE schedules for resident and nonresident taxpayers. The `get_paye_schedule()` function selects a schedule based on residency status and calculation date.

Effective-dated schedules help the system accommodate changes in tax rules over time. If no schedule applies to the requested date, the system raises `NoScheduleForDateError`.

The dates currently configured in this educational project are placeholders and should not be interpreted as verified legal effective dates.

In [30]:
income = Money.from_value(500000)
calculation_date = date(2026, 10, 18)

for residency in ["resident", "nonresident"]:
    context = TaxContext(
        income=income,
        effective_date=calculation_date,
        residency_status=residency,
    )

    result = PAYE().calculate(context)

    print(f"Residency: {residency.title()}")
    print(f"Taxable income: UGX {result.taxable_amount.amount:,.2f}")
    print(f"PAYE: UGX {result.amount.amount:,.2f}")
    print(f"Notes: {result.notes}")
    print("-" * 50)

Residency: Resident
Taxable income: UGX 500,000.00
PAYE: UGX 38,250.00
Notes: ('Calculated using the resident PAYE schedule effective from 2026-07-01',)
--------------------------------------------------
Residency: Nonresident
Taxable income: UGX 500,000.00
PAYE: UGX 75,500.00
Notes: ('Calculated using the nonresident PAYE schedule effective from 2026-07-01',)
--------------------------------------------------


In [31]:
# Test an unsupported date
from src.uganda_tax_engine.tax_schedules import get_paye_schedule

try:
    get_paye_schedule(
        date(2025, 6, 30),
        "resident",
    )
except NoScheduleForDateError as error:
    print("Expected schedule error handled successfully:")
    print(error)

Expected schedule error handled successfully:
No PAYE schedule configured for 'resident' on 2025-06-30.


7:  the payroll calculation

brings several components together to calculate payroll, including employee NSSF, employer NSSF, PAYE, Local Service Tax (LST), and net pay.

#### 8. Payroll Processing

The payroll module combines multiple deductions and contributions into a single calculation workflow.

Employee NSSF contributions are calculated at 5% of gross pay, while employer NSSF contributions are calculated at 10%. PAYE is calculated using the PAYE module, and Local Service Tax is supplied as a configurable amount.

Net pay is calculated by subtracting employee NSSF, PAYE, and Local Service Tax from gross pay. Employer NSSF is reported separately because it is an employer contribution rather than an employee deduction.

The rates and payroll rules used here are simplified project assumptions and should be verified against applicable requirements before practical use.

In [32]:
# Example monthly payroll calculation

payroll = Payroll()

gross_pay = Money.from_value(1500000)
local_service_tax = Money.from_value(5000)

payroll_result = payroll.calculate(
    gross_pay=gross_pay,
    effective_date=date(2026, 10, 18),
    residency_status="resident",
    local_service_tax=local_service_tax,
)

print(f"Gross pay:               {payroll_result.gross_pay}")
print(f"Employee NSSF (5%):      {payroll_result.employee_nssf}")
print(f"Employer NSSF (10%):     {payroll_result.employer_nssf}")
print(f"PAYE:                    {payroll_result.paye}")
print(f"Local Service Tax:       {payroll_result.local_service_tax}")
print(f"Net pay:                 {payroll_result.net_pay}")

Gross pay:               Money(UGX 1500000.00)
Employee NSSF (5%):      Money(UGX 75000.00)
Employer NSSF (10%):     Money(UGX 150000.00)
PAYE:                    Money(UGX 338250.00)
Local Service Tax:       Money(UGX 5000.00)
Net pay:                 Money(UGX 1081750.00)


In [33]:
# 7.3 a reconciliation check
expected_net_pay = (
    payroll_result.gross_pay.amount
    - payroll_result.employee_nssf.amount
    - payroll_result.paye.amount
    - payroll_result.local_service_tax.amount
)

assert payroll_result.net_pay.amount == max(
    expected_net_pay,
    Decimal("0.00"),
)

print("Payroll reconciliation passed.")

Payroll reconciliation passed.


This verifies that net pay reconciles with the deductions in your current implementation.

8: Value Added Tax (VAT)

demonstrates how your VAT module handles standard-rated, zero-rated, and exempt supplies, including the treatment of input VAT.

#### 9. Value Added Tax (VAT)

Value Added Tax is an indirect tax applied to taxable supplies. The tax engine distinguishes between standard-rated, zero-rated, and exempt supplies.

The implementation calculates output VAT, considers deductible input VAT, and determines the resulting net VAT payable or credit under the project's simplified rules.

- **Standard-rated supplies:** Apply the configured standard VAT rate.
- **Zero-rated supplies:** Apply a VAT rate of zero.
- **Exempt supplies:** Do not charge output VAT, and the current implementation disallows input VAT deduction for exempt supplies.

These calculations demonstrate the software design and are not a complete representation of every VAT rule or exception under Ugandan tax law.

In [34]:
# Compare VAT treatment across supply types

vat_calculator = VAT()

vat_examples = [
    {
        "supply_type": "standard",
        "income": "1000000",
        "input_vat": "50000",
    },
    {
        "supply_type": "zero_rated",
        "income": "1000000",
        "input_vat": "20000",
    },
    {
        "supply_type": "exempt",
        "income": "1000000",
        "input_vat": "20000",
    },
]

for example in vat_examples:
    context = TaxContext(
        income=Money.from_value(example["income"]),
        effective_date=date(2026, 10, 18),
        metadata={
            "supply_type": example["supply_type"],
            "input_vat": example["input_vat"],
        },
    )

    result = vat_calculator.calculate(context)

    print(f"Supply type: {example['supply_type']}")
    print(f"Taxable amount: UGX {result.taxable_amount.amount:,.2f}")
    print(f"Net VAT amount: UGX {result.amount.amount:,.2f}")
    print(f"Breakdown: {result.breakdown}")
    print("-" * 55)

Supply type: standard
Taxable amount: UGX 1,000,000.00
Net VAT amount: UGX 130,000.00
Breakdown: {'output_vat': Money(UGX 180000.00), 'input_vat': Money(UGX 50000.00), 'deductible_input_vat': Money(UGX 50000.00), 'net_vat_payable': Money(UGX 130000.00), 'vat_credit': Money(UGX 0.00)}
-------------------------------------------------------
Supply type: zero_rated
Taxable amount: UGX 0.00
Net VAT amount: UGX 0.00
Breakdown: {'output_vat': Money(UGX 0.00), 'input_vat': Money(UGX 20000.00), 'deductible_input_vat': Money(UGX 20000.00), 'net_vat_payable': Money(UGX 0.00), 'vat_credit': Money(UGX 20000.00)}
-------------------------------------------------------
Supply type: exempt
Taxable amount: UGX 0.00
Net VAT amount: UGX 0.00
Breakdown: {'output_vat': Money(UGX 0.00), 'input_vat': Money(UGX 20000.00), 'deductible_input_vat': Money(UGX 0.00), 'net_vat_payable': Money(UGX 0.00), 'vat_credit': Money(UGX 0.00)}
-------------------------------------------------------


In [35]:
# verify the VAT outcomes
# Verify the expected net VAT outcomes under the project's rules

expected_vat = {
    "standard": Decimal("130000.00"),
    "zero_rated": Decimal("0.00"),
    "exempt": Decimal("0.00"),
}

for supply_type, expected_amount in expected_vat.items():
    context = TaxContext(
        income=Money.from_value("1000000"),
        effective_date=date(2026, 10, 18),
        metadata={
            "supply_type": supply_type,
            "input_vat":(
                "50000" if supply_type == "standard" else "20000"
            ),
        },
    )

    result = vat_calculator.calculate(context)
    assert result.amount.amount == expected_amount

print("All three VAT scenario checks passed.")

All three VAT scenario checks passed.


zero-rated and exempt supplies are handled differently

engine correctly records deductible input VAT for zero-rated supplies but does not allow an input VAT deduction for exempt supplies under its current simplified rules.

Corporate tax

#### 7. Corporate Tax Calculation

This section demonstrates how the Uganda Tax Engine calculates corporate income tax using a simplified model. The taxable profit is calculated by deducting allowable business expenses from business revenue. Corporate tax is then calculated using the configured tax rate.

The example uses an assumed corporate tax rate of 30%. This is an educational assumption and should be verified against the applicable Uganda Revenue Authority guidance before real-world use.

In [36]:
corporate_tax_calculator = CorporateTax()

revenue = Money.from_value("100000000")
allowable_deductions = Money.from_value("40000000")

corporate_context = TaxContext(
    income=revenue,
    effective_date=date(2026, 10, 18),
    metadata={
        "allowable_deductions": allowable_deductions.amount,
    },
)

corporate_result = corporate_tax_calculator.calculate(corporate_context)

print(f"Business revenue: UGX {revenue.amount:,.2f}")
print(f"Allowable deductions: UGX {allowable_deductions.amount:,.2f}")
print(f"Taxable profit: UGX {corporate_result.taxable_amount.amount:,.2f}")
print(f"Corporate tax: UGX {corporate_result.amount.amount:,.2f}")
print(f"Breakdown: {corporate_result.breakdown}")

Business revenue: UGX 100,000,000.00
Allowable deductions: UGX 40,000,000.00
Taxable profit: UGX 60,000,000.00
Corporate tax: UGX 18,000,000.00
Breakdown: {'revenue': Money(UGX 100000000.00), 'allowable_deductions': Money(UGX 40000000.00), 'chargeable_income': Money(UGX 60000000.00), 'corporate_tax': Money(UGX 18000000.00)}


Verifying corporate tax calculations

In [37]:
# Verify corporate tax calculation
assert corporate_result.taxable_amount.amount == Decimal("60000000.00")
assert corporate_result.amount.amount == Decimal("18000000.00")

print("Corporate tax calculation checks passed.")

Corporate tax calculation checks passed.


Testing the tax engine

#### 8. Integrated Testing and Limitations

This section summarizes the verification performed on the tax engine. The notebook checks representative PAYE, payroll, VAT, and corporate tax calculations. The automated test suite provides additional unit and property-based tests for validation, boundary conditions, and calculation consistency.

The results demonstrate the implementation's behavior under the tested scenarios. They do not, by themselves, establish compliance with all current Ugandan tax laws. Tax rates, thresholds, effective dates, exemptions, and statutory rules must be verified against authoritative Uganda Revenue Authority guidance before practical use.

In [38]:
# Integrated verification of the examples run in this notebook

assert isinstance(paye_calculator, PAYE)
assert isinstance(vat_calculator, VAT)
assert isinstance(corporate_tax_calculator, CorporateTax)

assert corporate_result.taxable_amount.amount == Decimal("60000000.00")
assert corporate_result.amount.amount == Decimal("18000000.00")

assert all(
    result.amount.amount >= Decimal("0.00")
    for result in [corporate_result]
)

print("Integrated tax engine checks passed.")
print("PAYE, VAT, and corporate tax examples were executed.")
print("Review payroll reconciliation and all tax assumptions before deployment.")

Integrated tax engine checks passed.
PAYE, VAT, and corporate tax examples were executed.
Review payroll reconciliation and all tax assumptions before deployment.
